In [ ]:
from pathlib import Path

# Immutable GitHub source matching the verified native API and adaptive tuner.
CODE_COMMIT = "4cf894b470138fbc736403379b44c9dcefef78ad"
PREPARED_SOURCE_DIR = Path("/tmp") / ("multigpubeamsearch-" + CODE_COMMIT[:12])
MODEL_ROOT = Path("/tmp/cube555_inputs/model")
CHECKPOINT_PATH = MODEL_ROOT / "q555_f1_bell2k.pt"
RESMLP_CHECKPOINT_PATH = MODEL_ROOT / "q555_2k_BEST.pt"
PUZZLE_ID = 1020  # initial_state_id in test.csv; exactly one puzzle per run
BEAM_WIDTH = 1_048_576  # one explicit global beam; change only here
MAX_STEPS = 140
TRANSFORMER_WEIGHT = 0.8
NUM_GPUS = 2
TOUCH_BFS_RADIUS = 0  # use the adapter default; radius5 exceeds the default entry budget
RUN_TIMEOUT_SECONDS = 12 * 3600
CACHE_DIR = Path("/tmp/cube555_native_cache")
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "test_results/cube555_api_runs"
# This example never publishes results.
PUBLISH_RESULTS = False


# Cube555: CayleyPy + MultiGPUBeamSearch

Run the cells in order. Source, dependencies, checkpoints and puzzle inputs come
from GitHub. The source commit and input hashes are pinned and verified.
Choose the puzzle index, global beam and GPU count in the first cell; candidate
buffers and inference batches are selected automatically without changing that beam.

The explicit Transformer/ResMLP blend is0.8/0.2. The adaptive tuner measures
inference first, generates hardware-specific candidates, then refines around the
full-step winner. The first calibration takes minutes; compatible repeats use
cache with fresh memory admission. This is a bounded search, not a global optimum proof.

Default: index1020, beam1,048,576, two GPUs,140 steps. A clean 2xT4 run
of notebook version160 solved this puzzle in116 moves with independent replay.
The new persistent session is validated separately with short repeated searches;
that earlier full solve is not a rerun of the new session code.
[Validation scope](https://github.com/TryDotAtwo/MultiGPUBeamSearch/blob/4cf894b470138fbc736403379b44c9dcefef78ad/docs/validation/cube555-adaptive-20261025.md).

A returned solution is independently replayed. Results are saved in the output
directory; this notebook does not publish solutions automatically.

The search cell retains native_session across calls. Change start_state and rerun
it to reuse uploaded models, GPU buffers and processes. Changing beam, depth or
options safely recreates the workers. The final save cell closes the session.


In [ ]:
import subprocess, sys

if not (PREPARED_SOURCE_DIR / ".git").is_dir():
    PREPARED_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", str(PREPARED_SOURCE_DIR)], check=True)
    subprocess.run(["git", "-C", str(PREPARED_SOURCE_DIR), "remote", "add", "origin",
                    "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"], check=True)
    subprocess.run(["git", "-C", str(PREPARED_SOURCE_DIR), "fetch", "--depth", "1",
                    "origin", CODE_COMMIT], check=True)
    subprocess.run(["git", "-C", str(PREPARED_SOURCE_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(PREPARED_SOURCE_DIR),
    "rev-parse", "HEAD"], text=True).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError(f"Source commit mismatch: {actual_commit}")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "cayleypy @ https://github.com/cayleypy/cayleypy/archive/3cd11ecefc98e54d6fc0ccdec967d0bcc8ca4a93.tar.gz",
    str(PREPARED_SOURCE_DIR / "integrations/cayleypy_native")], check=True)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(f"Requested {NUM_GPUS} GPUs, available {torch.cuda.device_count()}; enable the GPU accelerator")
print("Source:", actual_commit)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])


In [ ]:
COMPETITION_ROOT = MODEL_ROOT.parent / "competition"
LAYOUT_PATH = MODEL_ROOT / "piece_layout_555.json"
# Download the public, pinned GitHub inputs before checking paths.
import hashlib, json, shutil, urllib.request, zipfile
INPUT_BUNDLE_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch/releases/download/cube555-inputs-20261001/cube555-inputs.zip"
INPUT_BUNDLE_SHA256 = "49d61a127540cd197db9bf75aacad5712a7ab54d3fbcb39bf2da9ae9bb477286"
input_root = MODEL_ROOT.parent.resolve()
input_root.mkdir(parents=True, exist_ok=True)
bundle = input_root.parent / "cube555-inputs.zip"
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
if not bundle.is_file() or file_sha256(bundle) != INPUT_BUNDLE_SHA256:
    print("Downloading Cube555 inputs from GitHub", flush=True)
    partial = bundle.with_suffix(".partial")
    with urllib.request.urlopen(INPUT_BUNDLE_URL, timeout=120) as response, partial.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    if file_sha256(partial) != INPUT_BUNDLE_SHA256:
        raise RuntimeError("Cube555 input ZIP SHA-256 mismatch")
    partial.replace(bundle)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        if not (input_root / member.filename).resolve().is_relative_to(input_root):
            raise RuntimeError("Unsafe input ZIP member")
    archive.extractall(input_root)
input_manifest = json.loads((input_root / "manifest.json").read_text())
for name, entry in input_manifest.items():
    path = (input_root / name).resolve()
    if not path.is_relative_to(input_root):
        raise RuntimeError("Unsafe input manifest path")
    if path.stat().st_size != entry["bytes"] or file_sha256(path) != entry["sha256"]:
        raise RuntimeError(f"Cube555 input verification failed: {name}")
print("GitHub inputs verified:", len(input_manifest), "files", flush=True)

for path in (CHECKPOINT_PATH, RESMLP_CHECKPOINT_PATH, LAYOUT_PATH,
             MODEL_ROOT / "puzzle_info.json", COMPETITION_ROOT / "test.csv",
             COMPETITION_ROOT / "sample_submission.csv"):
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}. Check the GitHub download or edit its path.")


In [ ]:
import csv, json, sys, subprocess, hashlib, uuid
from datetime import datetime, timezone
prepared_repo = Path(PREPARED_SOURCE_DIR).expanduser().resolve()
package_root = prepared_repo / "integrations/cayleypy_native"
if not (package_root / "multigpubeamsearch/__init__.py").is_file():
    raise RuntimeError("Run the GitHub setup cell first")
sys.path.insert(0, str(prepared_repo))
sys.path.insert(0, str(package_root))
from cayleypy import CayleyGraph, CayleyGraphDef
from multigpubeamsearch import NativeEnsemble, NativeOptions, NativeSession
from multigpubeamsearch.contracts import GraphContract
from tools.cube555.export import export_native_models

# JSON object order is the trained Q-head move order: never sort or add inverses.
info = json.loads((MODEL_ROOT / "puzzle_info.json").read_text())
generators = info["generators"]
definition = CayleyGraphDef.create(list(generators.values()),
    generator_names=list(generators), central_state=info["central_state"])
graph = CayleyGraph(definition, device="cpu", random_seed=42)
with (COMPETITION_ROOT / "test.csv").open(newline="") as stream:
    selected = [row for row in csv.DictReader(stream) if int(row["initial_state_id"]) == PUZZLE_ID]
if len(selected) != 1:
    raise ValueError(f"Expected one CSV row for initial_state_id={PUZZLE_ID}; got {len(selected)}")
start_state = [int(value) for value in selected[0]["initial_state"].split(",")]
contract = GraphContract.from_graph(graph, start_state)
if not 0.0 <= TRANSFORMER_WEIGHT <= 1.0:
    raise ValueError("TRANSFORMER_WEIGHT must be finite and in [0, 1]")
output = OUTPUT_ROOT / ("cube555_api_" + uuid.uuid4().hex)
output.mkdir(parents=True, exist_ok=False)
try:
    source_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=prepared_repo, text=True).strip()
    source_diff = subprocess.check_output(["git", "diff", "HEAD", "--", "integrations/cayleypy_native", "tools/cube555"], cwd=prepared_repo)
    source_status = subprocess.check_output(["git", "status", "--porcelain", "--", "integrations/cayleypy_native", "tools/cube555"], cwd=prepared_repo, text=True)
except (OSError, subprocess.CalledProcessError):
    source_head, source_diff, source_status = None, b"", "unavailable"
# Record the pinned commit and source hashes used by this run.
if source_head != CODE_COMMIT:
    raise RuntimeError("Source changed after setup")
provenance = dict(source_mode="github_pinned", release_status="published", expected_source_commit=CODE_COMMIT,
    prepared_source=str(prepared_repo), source_head=source_head, source_status=source_status,
    tracked_diff_sha256=hashlib.sha256(source_diff).hexdigest(),
    python_source_sha256={str(path.relative_to(prepared_repo)): hashlib.sha256(path.read_bytes()).hexdigest()
        for tree in (package_root / "multigpubeamsearch", prepared_repo / "tools/cube555")
        for path in sorted(tree.rglob("*.py"))},
    utc=datetime.now(timezone.utc).isoformat(), graph_hash=contract.graph_hash,
    puzzle_id=PUZZLE_ID, requested_beam_width=BEAM_WIDTH, max_steps=MAX_STEPS,
    input_bundle_url=INPUT_BUNDLE_URL, input_bundle_sha256=INPUT_BUNDLE_SHA256,
    input_files=input_manifest, blend=[TRANSFORMER_WEIGHT, 1.0-TRANSFORMER_WEIGHT])
(output / "provenance.json").write_text(json.dumps(provenance, indent=2))
# Export validates the graph/layout ordering and source checkpoints before native use.
transformer_head, resmlp_head = export_native_models(
    CHECKPOINT_PATH, RESMLP_CHECKPOINT_PATH, LAYOUT_PATH,
    MODEL_ROOT / "puzzle_info.json", output / "models",
    contract=GraphContract.from_graph(graph, graph.definition.central_state))
predictor = NativeEnsemble((transformer_head, resmlp_head),
    (TRANSFORMER_WEIGHT, 1.0 - TRANSFORMER_WEIGHT))


In [ ]:
options = NativeOptions(source_dir=prepared_repo, cache_dir=CACHE_DIR,
    num_gpus=NUM_GPUS, autotune=True, touch_bfs_radius=TOUCH_BFS_RADIUS,
    calibration_discover_capacity=True, calibration_full_frontier=True,
    calibration_pipeline_seconds=1800,
    timeout_seconds=RUN_TIMEOUT_SECONDS)
# Rerun this cell with another start before the final save/close cell.
if "native_session" not in globals() or native_session.closed:
    native_session = NativeSession()
result = native_session.beam_search(graph, native_options=options, start_state=start_state, predictor=predictor,
    backend="native", beam_width=BEAM_WIDTH, max_steps=MAX_STEPS,
    beam_mode="simple", use_child_scores=True, return_path=True)

calibration = result.native_metadata["profile"]["pipeline_calibration"]
print("Profile:", calibration["selection"], "cache:", calibration.get("cache_hit", False))
print("Search stop:", calibration.get("adaptive_search", {}).get("stop_reason"))


In [ ]:
# Replay independently of the adapter before saving a solution.
path = list(result.path) if result.path_found and result.path is not None else None
replay_valid = None
if result.path_found:
    if path is None:
        raise RuntimeError("Native API reported a solution without a replayable path")
    replay_state = list(start_state)
    ordered_moves = list(generators.values())
    for move in path:
        if type(move) is not int or not 0 <= move < len(ordered_moves):
            raise RuntimeError("Invalid move index in returned path")
        replay_state = [replay_state[index] for index in ordered_moves[move]]
    replay_valid = replay_state == info["central_state"]
    if not replay_valid:
        raise RuntimeError("Independent Cube555 replay failed")
metadata = result.native_metadata
effective_beam = metadata.get("effective_beam_width")
if effective_beam is not None and effective_beam != BEAM_WIDTH:
    raise RuntimeError(f"Requested beam {BEAM_WIDTH} was changed to {effective_beam}; refusing to report an equivalent run")
summary = dict(puzzle_id=PUZZLE_ID, path_found=result.path_found,
    path_length=len(path) if path is not None else None, path=path,
    move_names=[list(generators)[move] for move in path] if path is not None else None,
    replay_valid=replay_valid, requested_beam_width=BEAM_WIDTH,
    effective_beam_width=metadata.get("effective_beam_width"),
    max_steps=MAX_STEPS, native_metadata=metadata)
(output / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print("Saved:", output)

# End the GPU session after saving; repeated searches belong before this cell.
native_session.close()
